RNN Part 2
SimpleRNN, Long Short Term Memory (LSTM) and Gated Recurrent Unit (GRU)

In [37]:
import numpy as np
import matplotlib.pyplot as mat
import pandas as pd
import seaborn as sns

In [38]:
import tensorflow as tf
from tensorflow import keras
from keras.models import Model
from keras.layers import Input, Dense, Dropout, Embedding, SimpleRNN, LSTM, GRU
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.utils import to_categorical

I will perform RNN in whole text dataframe this time.

In [39]:
df= pd.read_csv(r"L:\4th sem\Mission Scopus\ML Datasets\quote_dataset.csv")

In [40]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [41]:
df['quote'] [0]

'“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'

In [42]:
df.shape

(3025, 2)

In [43]:
quotes = df['quote']
quotes.head()

0    “The world as we have created it is a process ...
1    “It is our choices, Harry, that show what we t...
2    “There are only two ways to live your life. On...
3    “The person, be it gentleman or lady, who has ...
4    “Imperfection is beauty, madness is genius and...
Name: quote, dtype: str

Data Cleaning (Converting all the texts into lower case and removing all the punctuations)

In [44]:
quotes= quotes.str.lower()

In [45]:
import string
quotes = quotes.str.translate(str.maketrans('', '', string.punctuation))

In [46]:
quotes = quotes.str.replace(r"[“”‘’]", "", regex=True)

In [47]:
quotes.head()

0    the world as we have created it is a process o...
1    it is our choices harry that show what we trul...
2    there are only two ways to live your life one ...
3    the person be it gentleman or lady who has not...
4    imperfection is beauty madness is genius and i...
Name: quote, dtype: str

In [57]:
# Longest original quote in characters
print(df['quote'].str.len().max())

# Approximate number of words in the longest quote (after cleaning)
print(quotes.str.split().str.len().max())

# Exact token length of the longest sequence
print(max(len(seq) for seq in sequences))

3903
746
746


In order to guess the vocab_size, I will find the number of unique words in the dataset.

In [48]:
vocab_size= 10000
tokenizer= Tokenizer(num_words= vocab_size, oov_token= 'OOV')
tokenizer.fit_on_texts(quotes)

In [49]:
word_index= tokenizer.word_index
print(len(word_index))
list(word_index.items())[:10]

8472


[('OOV', 1),
 ('the', 2),
 ('you', 3),
 ('to', 4),
 ('and', 5),
 ('a', 6),
 ('i', 7),
 ('is', 8),
 ('of', 9),
 ('that', 10)]

In [50]:
sequences= tokenizer.texts_to_sequences(quotes)

In [51]:
for i in range(3):
    print(quotes[i])
    print(sequences[i])

the world as we have created it is a process of our thinking it cannot be changed without changing our thinking
[2, 63, 30, 20, 17, 934, 11, 8, 6, 1138, 9, 71, 286, 11, 147, 13, 802, 106, 742, 71, 286]
it is our choices harry that show what we truly are far more than our abilities
[11, 8, 71, 861, 374, 10, 435, 22, 20, 466, 15, 297, 53, 55, 71, 2402]
there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle
[43, 15, 54, 202, 707, 4, 82, 16, 37, 38, 8, 30, 321, 96, 8, 6, 1027, 2, 103, 8, 30, 321, 129, 8, 6, 1027]


Create multiple sequences/ sequence from multiple words

In [52]:
X= []
y= []
for seq in sequences:
    for i in range(1, len(seq)):
        input_seq= seq[:i]
        output_seq= seq[i]
        X.append(input_seq)
        y.append(output_seq)

In [53]:
X

[[2],
 [2, 63],
 [2, 63, 30],
 [2, 63, 30, 20],
 [2, 63, 30, 20, 17],
 [2, 63, 30, 20, 17, 934],
 [2, 63, 30, 20, 17, 934, 11],
 [2, 63, 30, 20, 17, 934, 11, 8],
 [2, 63, 30, 20, 17, 934, 11, 8, 6],
 [2, 63, 30, 20, 17, 934, 11, 8, 6, 1138],
 [2, 63, 30, 20, 17, 934, 11, 8, 6, 1138, 9],
 [2, 63, 30, 20, 17, 934, 11, 8, 6, 1138, 9, 71],
 [2, 63, 30, 20, 17, 934, 11, 8, 6, 1138, 9, 71, 286],
 [2, 63, 30, 20, 17, 934, 11, 8, 6, 1138, 9, 71, 286, 11],
 [2, 63, 30, 20, 17, 934, 11, 8, 6, 1138, 9, 71, 286, 11, 147],
 [2, 63, 30, 20, 17, 934, 11, 8, 6, 1138, 9, 71, 286, 11, 147, 13],
 [2, 63, 30, 20, 17, 934, 11, 8, 6, 1138, 9, 71, 286, 11, 147, 13, 802],
 [2, 63, 30, 20, 17, 934, 11, 8, 6, 1138, 9, 71, 286, 11, 147, 13, 802, 106],
 [2,
  63,
  30,
  20,
  17,
  934,
  11,
  8,
  6,
  1138,
  9,
  71,
  286,
  11,
  147,
  13,
  802,
  106,
  742],
 [2,
  63,
  30,
  20,
  17,
  934,
  11,
  8,
  6,
  1138,
  9,
  71,
  286,
  11,
  147,
  13,
  802,
  106,
  742,
  71],
 [11],
 [11, 8],
 [11

In [54]:
y

[63,
 30,
 20,
 17,
 934,
 11,
 8,
 6,
 1138,
 9,
 71,
 286,
 11,
 147,
 13,
 802,
 106,
 742,
 71,
 286,
 8,
 71,
 861,
 374,
 10,
 435,
 22,
 20,
 466,
 15,
 297,
 53,
 55,
 71,
 2402,
 15,
 54,
 202,
 707,
 4,
 82,
 16,
 37,
 38,
 8,
 30,
 321,
 96,
 8,
 6,
 1027,
 2,
 103,
 8,
 30,
 321,
 129,
 8,
 6,
 1027,
 117,
 13,
 11,
 2403,
 33,
 1028,
 32,
 83,
 14,
 599,
 12,
 6,
 75,
 1313,
 121,
 13,
 2404,
 330,
 8,
 307,
 743,
 8,
 637,
 5,
 40,
 145,
 4,
 13,
 678,
 1314,
 55,
 678,
 744,
 14,
 4,
 203,
 6,
 93,
 9,
 436,
 284,
 203,
 6,
 93,
 9,
 518,
 8,
 145,
 4,
 13,
 1315,
 18,
 22,
 3,
 15,
 55,
 4,
 13,
 178,
 18,
 22,
 3,
 15,
 14,
 17,
 14,
 1316,
 174,
 52,
 417,
 2406,
 707,
 10,
 313,
 274,
 184,
 8,
 41,
 6,
 803,
 1317,
 3,
 48,
 51,
 60,
 322,
 11,
 8,
 170,
 40,
 12,
 638,
 482,
 112,
 106,
 1029,
 8,
 41,
 3,
 51,
 275,
 37,
 8,
 22,
 3,
 66,
 11,
 49,
 185,
 22,
 86,
 133,
 4,
 745,
 59,
 127,
 40,
 6,
 1895,
 177,
 19,
 2,
 75,
 211,
 8,
 3,
 97,
 4,
 467,
 60,
 86,

In [55]:
len(X), len(y)

(84823, 84823)

In [56]:
max_length= max(len(x) for x in X)
max_length

745

In [58]:
X_pad= pad_sequences(X, maxlen= max_length, padding= "pre")
y= np.array(y)

In [59]:
X_pad.shape, y.shape

((84823, 745), (84823,))

In [60]:
embed_dim= 32
rnn_units= 64

In [61]:
input_layer= Input(shape= (max_length,), dtype= "int32", name= "Input_Layer")
embedding_layer= Embedding(input_dim= vocab_size, output_dim= embed_dim, mask_zero= True, name= "Embedding_Layer")(input_layer)

Model Creation and Training for SimpleRNN

In [62]:
rnn= SimpleRNN(units= rnn_units, return_sequences= False, return_state= False, name= "SimpleRNN_Layer")
embedding_layer_last= rnn(embedding_layer)
output_layer= Dense(units= vocab_size, activation= "softmax", name= "Output_Layer")(embedding_layer_last)

c:\Users\DELL\AppData\Local\Programs\Python\Python314\Lib\site-packages\keras\src\layers\layer.py:1062: UserWarning: Layer 'SimpleRNN_Layer' (of type SimpleRNN) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


In [63]:
model= Model(inputs= input_layer, outputs= output_layer)

In [67]:
model.compile(optimizer= "adam", loss= "sparse_categorical_crossentropy", metrics= ["accuracy"])

In [68]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Input_Layer (InputLayer)        │ (None, 745)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Embedding_Layer (Embedding)     │ (None, 745, 32)        │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ SimpleRNN_Layer (SimpleRNN)     │ (None, 64)             │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output_Layer (Dense)            │ (None, 10000)          │       650,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 976,208 (3.72 MB)

 Trainable params: 976,208 (3.72 MB)

 Non-trainable params: 0 (0.00 B)

In [69]:
simple_rnn_model= model.fit(X_pad, y, epochs= 10, batch_size= 128, validation_split= 0.2, verbose= 1)

Epoch 1/10
531/531 ━━━━━━━━━━━━━━━━━━━━ 307s 572ms/step - accuracy: 0.0368 - loss: 6.8750 - val_accuracy: 0.0392 - val_loss: 6.7255
Epoch 2/10
531/531 ━━━━━━━━━━━━━━━━━━━━ 301s 566ms/step - accuracy: 0.0483 - loss: 6.4115 - val_accuracy: 0.0545 - val_loss: 6.6483
Epoch 3/10
531/531 ━━━━━━━━━━━━━━━━━━━━ 296s 557ms/step - accuracy: 0.0679 - loss: 6.1522 - val_accuracy: 0.0765 - val_loss: 6.4821
Epoch 4/10
531/531 ━━━━━━━━━━━━━━━━━━━━ 389s 733ms/step - accuracy: 0.0895 - loss: 5.9018 - val_accuracy: 0.0858 - val_loss: 6.4118
Epoch 5/10
476/531 ━━━━━━━━━━━━━━━━━━━━ 44s 811ms/step - accuracy: 0.1025 - loss: 5.7272

: 

Since, the kernel is crashed due to lack of compatibility of CPU. I will try it from google colab and use epochs 100 and download h5 file of training and try in this environment.

Model Creation and Training for LSTM

In [ ]:
rnn2= LSTM(units= rnn_units, return_sequences= False, return_state= False, name= "LSTM_Layer")
embedding_layer_last2= rnn2(embedding_layer)
output_layer2= Dense(units= vocab_size, activation= "softmax", name= "Output_Layer2")(embedding_layer_last2)

In [ ]:
model2= Model(inputs= input_layer, outputs= output_layer2)

In [ ]:
model2.compile(optimizer= "adam", loss= "sparse_categorical_crossentropy", metrics= ["accuracy"])

In [ ]:
model2.summary()

In [ ]:
lstm_model= model2.fit(X_pad, y, epochs= 10, batch_size= 128, validation_split= 0.2, verbose= 1)

Model Creation and Training for GRU

In [ ]:
rnn3= GRU(units= rnn_units, return_sequences= False, return_state= False, name= "GRU_Layer")
embedding_layer_last3= rnn3(embedding_layer)
output_layer3= Dense(units= vocab_size, activation= "softmax", name= "Output_Layer3")(embedding_layer_last3)

In [ ]:
model3= Model(inputs= input_layer, outputs= output_layer3)

In [ ]:
model3.compile(optimizer= "adam", loss= "sparse_categorical_crossentropy", metrics= ["accuracy"])

In [ ]:
model3.summary()

In [ ]:
gru_model= model3.fit(X_pad, y, epochs= 10, batch_size= 128, validation_split= 0.2, verbose= 1)